# Notebook 04.2: ResNet18 — Full Fine-Tune + Full TTA

Same training recipe as `04.1_resnet`, but with **multi-scale + hflip TTA** at inference.

- **Training**: identical to 04.1 — ResNet18, all layers unfrozen, `Adam(lr=1e-4)`, 8 epochs, IMG_SIZE=224.
- **Inference**: multi-scale `[224, 256, 288]` × hflip = **6 averaged views** per image.
- Used for both validation-set threshold selection *and* test-set submission so the threshold matches the inference regime.

### Why TTA helps
Averaging predictions over slightly different views of the same image reduces noise from specific resize interpolation and left-right asymmetries. Typical lift on small datasets: +0.5–1% AUROC, free at inference.

---
## Step 1: Imports

In [ ]:
import copy
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

print("Imports OK")

## Step 2: Reproducibility and Device

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

## Step 3: Configuration

Using `IMG_SIZE=224` — ResNet18's native pretraining resolution, same as Auste's notebook. Fast and fair comparison.

In [ ]:
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TRAIN_CSV = REPO / "data" / "train_val" / "train_val.csv"
TRAIN_IMG_DIR = REPO / "data" / "train_val" / "images"
TEST_IMG_DIR = REPO / "data" / "test_images"
PRED_DIR = REPO / "outputs" / "predictions"
CKPT_DIR = REPO / "outputs" / "checkpoints"
PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224          # ResNet18 native pretraining resolution
BATCH_SIZE = 16
NUM_WORKERS = 0
VAL_FRAC = 0.2
EPOCHS = 8
LR = 1e-4

# Multi-scale TTA: inference runs at these sizes, each averaged with its hflip.
TTA_SCALES = [IMG_SIZE, IMG_SIZE + 32, IMG_SIZE + 64]   # [224, 256, 288]
TTA_HFLIP = True

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"IMG_SIZE  : {IMG_SIZE}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"EPOCHS    : {EPOCHS}")
print(f"LR        : {LR}")
print(f"TTA scales: {TTA_SCALES}  (hflip={TTA_HFLIP})")

## Step 4: Load labels and inspect class balance

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df = df.rename(columns={
    "Image Index": "image_file",
    "Patient Age": "age",
    "Patient Sex": "sex",
    "Finding Labels": "finding",
})
df["label"] = (df["finding"] == "Cardiomegaly").astype(int)

print(df.head())
print("\nFinding counts:")
print(df["finding"].value_counts())
print(f"\nPositive rate: {df['label'].mean():.3f}")
print(f"Total         : {len(df)}")

## Step 5: Dataset class

In [ ]:
class CardiomegalyDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, return_label=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.return_label = return_label

    def __len__(self):
        return len(self.df)

    def _load_image(self, fname):
        img = Image.open(self.img_dir / fname)
        if img.mode != "RGB":
            img = img.convert("RGB")
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row["image_file"])
        if self.transform is not None:
            img = self.transform(img)
        if not self.return_label:
            return img, row["image_file"]
        return img, torch.tensor(row["label"], dtype=torch.float32)

## Step 6: Stratified train / validation split

80/20, stratified on the label. `SEED=42` matches every other notebook, so the same images end up in train vs. val across 01, 02, 03, 04.

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=VAL_FRAC, stratify=df["label"], random_state=SEED
)
print(f"Train: {len(train_df)}  (pos rate {train_df['label'].mean():.3f})")
print(f"Val  : {len(val_df)}  (pos rate {val_df['label'].mean():.3f})")

## Step 7: Transforms

In [ ]:
def build_transform(img_size, augment=False):
    ops = []
    if img_size is not None:
        ops.append(T.Resize((img_size, img_size)))
    if augment:
        ops += [
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=5),
            T.ColorJitter(brightness=0.1, contrast=0.1),
        ]
    ops += [T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]
    return T.Compose(ops)

# Training uses augmentation; val uses clean resize only.
train_tf = build_transform(IMG_SIZE, augment=True)
val_tf = build_transform(IMG_SIZE, augment=False)

train_ds = CardiomegalyDataset(train_df, TRAIN_IMG_DIR, transform=train_tf)
val_ds = CardiomegalyDataset(val_df, TRAIN_IMG_DIR, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

---
## Metrics helpers

In [ ]:
def sens_spec(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    pred = (y_prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return sens, spec

def datathon_score(auroc, sens, spec):
    return 0.5 * auroc + 0.25 * sens + 0.25 * spec

def best_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(thr[np.argmax(tpr - fpr)])

## Train / Eval loops

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ys, ps = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs).squeeze(1)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        ys.extend(labels.numpy().tolist())
        ps.extend(probs.tolist())
    ys = np.array(ys); ps = np.array(ps)
    auroc = roc_auc_score(ys, ps)
    sens, spec = sens_spec(ys, ps, threshold=0.5)
    return {"auroc": auroc, "sens": sens, "spec": spec, "y": ys, "p": ps}

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    pbar = tqdm(loader, leave=False)
    for imgs, labels in pbar:
        imgs = imgs.to(device); labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total += loss.item()
        pbar.set_description(f"loss {loss.item():.3f}")
    return total / len(loader)

def train(model, train_loader, val_loader, optimizer, epochs, device, tag="model"):
    criterion = nn.BCEWithLogitsLoss()
    best = {"auroc": -1.0, "state": None, "epoch": 0}
    history = []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val = evaluate(model, val_loader, device)
        dt = time.time() - t0
        score = datathon_score(val["auroc"], val["sens"], val["spec"])
        history.append({"epoch": epoch, "loss": tr_loss, **{k: val[k] for k in ("auroc", "sens", "spec")}, "score": score})
        star = ""
        if val["auroc"] > best["auroc"]:
            best = {"auroc": val["auroc"], "state": copy.deepcopy(model.state_dict()), "epoch": epoch}
            star = " ★"
            torch.save(model.state_dict(), CKPT_DIR / f"{tag}_best.pt")
        print(
            f"[{tag}] ep {epoch:02d}/{epochs}  loss {tr_loss:.4f}  "
            f"AUROC {val['auroc']:.4f}  sens {val['sens']:.3f}  spec {val['spec']:.3f}  "
            f"score {score:.4f}  ({dt:.1f}s){star}"
        )
    model.load_state_dict(best["state"])
    return model, history, best

---
## Full TTA helper (multi-scale + hflip)

Runs the model at multiple image sizes + optional hflip, averages sigmoid probabilities. Used for val-set threshold tuning *and* test inference so the threshold matches the submission regime.

In [ ]:
@torch.no_grad()
def predict_probs_tta(model, loader, device, tta_scales=None, tta_hflip=False, return_labels=False):
    """Average sigmoid probabilities across multi-scale + hflip views."""
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        imgs = batch[0].to(device)
        scales = tta_scales if tta_scales else [imgs.shape[-1]]
        prob_sum = torch.zeros(imgs.size(0), device=device)
        n_views = 0
        for s in scales:
            view = imgs if s == imgs.shape[-1] else F.interpolate(imgs, size=(s, s), mode="bilinear", align_corners=False)
            prob_sum = prob_sum + torch.sigmoid(model(view).squeeze(1))
            n_views += 1
            if tta_hflip:
                prob_sum = prob_sum + torch.sigmoid(model(torch.flip(view, dims=[3])).squeeze(1))
                n_views += 1
        all_probs.append((prob_sum / n_views).cpu().numpy())
        if return_labels:
            all_labels.append(batch[1].numpy())
    probs = np.concatenate(all_probs)
    if return_labels:
        return probs, np.concatenate(all_labels)
    return probs


def evaluate_tta(model, loader, device, tta_scales=None, tta_hflip=False):
    """evaluate() but with TTA. Returns same dict shape."""
    p, y = predict_probs_tta(model, loader, device, tta_scales=tta_scales, tta_hflip=tta_hflip, return_labels=True)
    auroc = roc_auc_score(y, p)
    sens, spec = sens_spec(y, p, threshold=0.5)
    return {"auroc": auroc, "sens": sens, "spec": spec, "y": y, "p": p}

---
## Train: Full fine-tune

Replace `model.fc` with a single `Linear(512, 1)` (bare, no Dropout — matches Auste's recipe). Unfreeze **all** parameters and train with `Adam(lr=1e-4)`.

In [ ]:
def build_full_finetune():
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    # All params remain trainable (no freezing).
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

model = build_full_finetune().to(device)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.2f}%)")

optimizer = optim.Adam(model.parameters(), lr=LR)
model, history, best = train(
    model, train_loader, val_loader, optimizer, epochs=EPOCHS, device=device, tag="resnet18_fullft"
)

---
## Validation ROC + threshold tuning

Single model now (not 3 stages), so we directly evaluate it, plot the ROC, and pick a threshold.

In [ ]:
val = evaluate_tta(model, val_loader, device, tta_scales=TTA_SCALES, tta_hflip=TTA_HFLIP)
fpr, tpr, _ = roc_curve(val["y"], val["p"])

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, lw=2, label=f"full fine-tune + TTA  AUROC={val['auroc']:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Validation ROC — ResNet18 full fine-tune (with TTA)")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## Best threshold + final score

Grid-search threshold 0.1 → 0.9 in 0.05 steps, pick the one maximizing the datathon score (same approach Auste used — for datathon scoring this is equivalent to Youden's J but directly interpretable).

In [ ]:
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []
for t in thresholds:
    s, sp = sens_spec(val["y"], val["p"], threshold=t)
    rows.append({"threshold": t, "auc": val["auroc"], "sens": s, "spec": sp,
                 "score": datathon_score(val["auroc"], s, sp)})
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

best_row = results_df.loc[results_df["score"].idxmax()]
best_thr = float(best_row["threshold"])
best_score = float(best_row["score"])
print(f"\nBest threshold: {best_thr:.2f}  score: {best_score:.4f}")

---
## Generate submission CSV for the 176 test images

In [ ]:
test_files = sorted(p.name for p in TEST_IMG_DIR.iterdir() if p.suffix.lower() == ".png")
print(f"Test images: {len(test_files)}")

test_df = pd.DataFrame({"image_file": test_files, "label": 0})
test_tf = build_transform(IMG_SIZE, augment=False)
test_ds = CardiomegalyDataset(test_df, TEST_IMG_DIR, transform=test_tf, return_label=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Full multi-scale + hflip TTA for submission
test_probs = predict_probs_tta(model, test_loader, device, tta_scales=TTA_SCALES, tta_hflip=TTA_HFLIP)
all_names = [name for _, name in test_ds]

sub = pd.DataFrame({"image_file": all_names, "prob": test_probs.tolist()})
sub["pred"] = (sub["prob"] >= best_thr).astype(int)
sub = sub.sort_values("image_file").reset_index(drop=True)

stamp = time.strftime("%Y%m%d_%H%M")
out_path = PRED_DIR / f"submission_resnet18_fullft_fullTTA_{stamp}.csv"
sub.to_csv(out_path, index=False)
print(f"\nWrote {out_path}")
print(sub.head())
print(f"\nPositive rate in submission: {sub['pred'].mean():.3f}")